# PC-RadNet: figure generation from stored results

Plots of segmentation, grading, disorder and benchmark results computed from the files in `results/`.
Run from the repository root or from `notebooks/`. Each figure is written as PNG and PDF to `results/figures/`.


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# Cell 1: Environment, Style Setup & Data Loading
# ══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
import json, os, shutil, glob, warnings, textwrap
from pathlib import Path
from IPython.display import display, Image as IPImage
from PIL import Image as PILImage
warnings.filterwarnings('ignore')

# ── Plot style ────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.03,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'lines.linewidth': 1.0,
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'text.usetex': False,
    'mathtext.fontset': 'dejavuserif',
    'legend.framealpha': 0.9,
    'legend.edgecolor': '#cccccc',
    'legend.borderpad': 0.4,
})

# ── Paths (portable: relative to workspace root) ────────────────────────
ROOT = Path.cwd()
if ROOT.name in ('scripts', 'notebooks'):
    ROOT = ROOT.parent
RESULTS    = str(ROOT / 'results')
# FIGURES    = str(ROOT / 'figures')
FIGURES_D  = str(ROOT / 'results' / 'figures')
# MANUSCRIPT = str(ROOT / 'manuscript')
SCRIPTS    = str(ROOT / 'scripts')
import sys; sys.path.insert(0, str(ROOT / 'scripts'))
from repo_config import load_config
DATASET    = load_config()['paths']['dataset_root']
# os.makedirs(FIGURES, exist_ok=True)
os.makedirs(FIGURES_D, exist_ok=True)

# ── Colour palette (colorblind-safe, Tol bright) ─────────────────────────
PAL = {
    'blue':   '#3274A1',
    'orange': '#E1812C',
    'green':  '#3A923A',
    'red':    '#C03D3E',
    'purple': '#9372B2',
    'grey':   '#808080',
    'teal':   '#17BECF',
    'brown':  '#8C564B',
}
ABL_COLORS = {'A0': PAL['grey'], 'A1': PAL['blue'],
              'A2': PAL['green'], 'A3': PAL['red']}
GRADE_COLORS = [PAL['purple'], PAL['blue'], PAL['green'],
                PAL['orange'], PAL['red']]

# ── Load all data ─────────────────────────────────────────────────────────
df_a2 = pd.read_csv(os.path.join(RESULTS, 'test_A2_per_image.csv'))
df_a0 = pd.read_csv(os.path.join(RESULTS, 'test_A0_per_image.csv'))
df_a1 = pd.read_csv(os.path.join(RESULTS, 'test_A1_per_image.csv'))
df_a3 = pd.read_csv(os.path.join(RESULTS, 'test_A3_per_image.csv'))

with open(os.path.join(RESULTS, 'all_ablation_metrics_test.json')) as f:
    abl_metrics = json.load(f)
with open(os.path.join(RESULTS, 'all_ablation_metrics_val.json')) as f:
    abl_metrics_val = json.load(f)
with open(os.path.join(RESULTS, 'bootstrap_cis_all.json')) as f:
    bootstrap_cis = json.load(f)
with open(os.path.join(RESULTS, 'classification_report_grading.json')) as f:
    cls_report = json.load(f)
with open(os.path.join(RESULTS, 'baseline_metrics_test.json')) as f:
    baseline_metrics_test = json.load(f)
with open(os.path.join(RESULTS, 'class_weights.json')) as f:
    class_wts = json.load(f)
with open(os.path.join(RESULTS, 'phase2_A2_metrics.json')) as f:
    phase2_info = json.load(f)
with open(os.path.join(RESULTS, 'prompt_gate_analysis.json')) as f:
    gate_data = json.load(f)

cm_raw      = np.loadtxt(os.path.join(RESULTS, 'confusion_matrix_grading.csv'),
                         delimiter=',', skiprows=1)
stat_df     = pd.read_csv(os.path.join(RESULTS, 'statistical_tests.csv'))
baseline_df = pd.read_csv(os.path.join(RESULTS, 'baseline_comparison.csv'))
sam_test_df = pd.read_csv(os.path.join(RESULTS, 'baseline_sam_test.csv'))
sam_val_df  = pd.read_csv(os.path.join(RESULTS, 'baseline_sam_val.csv'))
ref_reports = pd.read_csv(os.path.join(DATASET, 'structured_reports.csv'))
split_df    = pd.read_csv(os.path.join(RESULTS, 'split_report.csv'))

# ── Helpers ──────────────────────────────────────────────────────────────
def save_fig(fig, name, disp_max_w=1600):
    """Save figure as 300-dpi PNG + PDF to figures/; create resized display
    thumbnail in figures_display/; copy PNG to manuscript/.
    Works correctly even after plt.close() because PIL reads the saved file."""
    # png_path  = os.path.join(FIGURES,    f'{name}.png')
    # pdf_path  = os.path.join(FIGURES,    f'{name}.pdf')
    disp_path = os.path.join(FIGURES_D,  f'{name}.png')
    # mss_path  = os.path.join(MANUSCRIPT, f'{name}.png')
    # Save high-res outputs
    fig.savefig(png_path, bbox_inches='tight', pad_inches=0.03, dpi=300)
    fig.savefig(pdf_path, bbox_inches='tight', pad_inches=0.03)
    shutil.copy2(png_path, mss_path)
    # Create display-resolution thumbnail for show()
    img = PILImage.open(png_path).convert('RGB')
    if img.width > disp_max_w:
        ratio = disp_max_w / img.width
        img   = img.resize((disp_max_w, int(img.height * ratio)), PILImage.LANCZOS)
    img.save(disp_path, 'PNG', optimize=True)
    plt.close(fig)
    print(f'  ✓ Saved: {name}.png / .pdf  [disp: {img.width}×{img.height}px]')

def show(stem, width=800):
    """Display resized PNG from figures_display/ by filepath string only.
    Never reads figures/ originals."""
    path = os.path.join(FIGURES_D, f'{stem}.png')
    if os.path.exists(path):
        display(IPImage(filename=path, width=width))
    else:
        print(f'[MISSING IN FIGURES_D] {stem}.png — run save_fig first')

# ── Quick verification ────────────────────────────────────────────────────
n_test = len(df_a2)
n_ax   = (df_a2['plane'] == 'axial').sum()
n_sag  = (df_a2['plane'] == 'sagittal').sum()
print(f'Test images: {n_test}  (axial={n_ax}, sagittal={n_sag})')
print(f'Ablation variants loaded: {[m["ablation"] for m in abl_metrics]}')
print(f'Split: train={split_df[split_df.Split=="train"].shape[0]}, '
      f'val={split_df[split_df.Split=="val"].shape[0]}, '
      f'test={split_df[split_df.Split=="test"].shape[0]} patients')
print(f'FIGURES_D: {FIGURES_D}  exists={os.path.exists(FIGURES_D)}')
print(f'figures_display PNGs: {len(glob.glob(os.path.join(FIGURES_D,"*.png")))}')
print('Setup complete.')


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# Cell 2: Data Integrity QA + Canonical Ablation Ordering
# ══════════════════════════════════════════════════════════════════════
expected_abl = ['A0', 'A1', 'A2', 'A3']

# Enforce deterministic ordering to prevent silent plotting mismatches.
abl_test_map = {m['ablation']: m for m in abl_metrics}
abl_val_map = {m['ablation']: m for m in abl_metrics_val}

if set(abl_test_map.keys()) != set(expected_abl):
    raise ValueError(f'Unexpected test ablations: {sorted(abl_test_map.keys())}')
if set(abl_val_map.keys()) != set(expected_abl):
    raise ValueError(f'Unexpected val ablations: {sorted(abl_val_map.keys())}')

abl_metrics = [abl_test_map[a] for a in expected_abl]
abl_metrics_val = [abl_val_map[a] for a in expected_abl]

# Cross-check aggregate Dice/IoU against per-image CSV means (no made-up values).
per_image_map = {
    'A0': df_a0,
    'A1': df_a1,
    'A2': df_a2,
    'A3': df_a3,
}

# Tolerance accounts for JSON float serialization precision
tol = 1e-4
qa_rows = []
for a in expected_abl:
    df_cur = per_image_map[a]
    m_json = abl_test_map[a]
    dice_csv = float(df_cur['prompted_dice'].mean())
    iou_csv = float(df_cur['prompted_iou'].mean())
    dice_json = float(m_json['prompted_dice_mean'])
    iou_json = float(m_json['prompted_iou_mean'])
    dice_ok = abs(dice_csv - dice_json) <= tol
    iou_ok = abs(iou_csv - iou_json) <= tol
    qa_rows.append({
        'ablation': a,
        'dice_csv': dice_csv,
        'dice_json': dice_json,
        'dice_match': dice_ok,
        'iou_csv': iou_csv,
        'iou_json': iou_json,
        'iou_match': iou_ok,
    })

qa_df = pd.DataFrame(qa_rows)
display(qa_df)

if not qa_df['dice_match'].all() or not qa_df['iou_match'].all():
    raise ValueError('QA failed: aggregate metrics do not match per-image CSV means.')

print('QA passed: all aggregate Dice/IoU values match per-image results within tolerance.')
print('Canonical ablation order fixed to:', expected_abl)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MASTER FIGURE MANIFEST  - tracks every figure in the manuscript
# ══════════════════════════════════════════════════════════════════════
manifest = [
    # stem, group, source_script, tex_label, caption_abbrev
    ("fig1_architecture",            "Main", "visualize.py (TikZ)",  "fig:arch",           "System architecture diagram"),
    ("fig2_training_curves",         "Main", "visualize.py",          "fig:training",       "Training/val loss curves"),
    ("fig3_qualitative",             "Main", "visualize.py (model)",  "fig:qualitative",    "Qualitative segmentation examples"),
    ("fig4_ablation_bars",           "Main", "visualize.py",          "fig:ablation",       "Ablation bar charts (val)"),
    ("fig5_confusion_matrix",        "Main", "visualize.py",          "fig:cm",             "Grading confusion matrix"),
    ("fig6_dataset_distribution",    "Main", "Section-F cell",        "fig:dataset_dist",   "Dataset grade & split distribution"),
    ("fig7_bland_altman",            "Main", "Section-F cell",        "fig:ba",             "Bland-Altman IVD height"),
    ("fig8_per_disc_level",          "Main", "Section-F cell",        "fig:disc_level",     "Per-disc-level seg performance"),
    ("fig9_roc_curves",              "Main", "Section-F cell",        "fig:roc",            "ROC curves (grading + disorder)"),
    ("fig10_grade_performance",      "Main", "Section-F cell",        "fig:grade_perf",     "Per-grade F1 & sample counts"),
    ("fig11_failure_analysis",       "Main", "Section-F cell",        "fig:failure",        "Failure case analysis"),
    ("fig_seg_perclass_heatmap",     "Supp-A","Section-A cell",       "fig:seg_heatmap",    "Per-class per-variant dice heatmap"),
    ("fig_seg_axial_vs_sagittal",    "Supp-A","Section-A cell",       "fig:seg_planes",     "Axial vs sagittal violin"),
    ("fig_seg_per_disc_level",       "Supp-A","Section-A cell",       "fig:disc_seg",       "Per-disc-level segmentation"),
    ("fig_seg_class_imbalance",      "Supp-A","Section-A cell",       "fig:imbalance",      "Class imbalance bar chart"),
    ("fig_seg_dice_distribution",    "Supp-A","Section-A cell",       "fig:dice_dist",      "Dice score distribution"),
    ("fig_grade_confusion_matrix",   "Supp-B","Section-B cell",       "fig:grade_cm",       "Full 5-class confusion matrix"),
    ("fig_grade_per_grade_metrics",  "Supp-B","Section-B cell",       "fig:grade_metrics",  "Per-grade P/R/F1"),
    ("fig_grade_error_distribution", "Supp-B","Section-B cell",       "fig:grade_err",      "Grading error distribution"),
    ("fig_grade_bland_altman",       "Supp-B","Section-B cell",       "fig:grade_ba",       "Grade Bland-Altman"),
    ("fig_report_roc_pr_curves",     "Supp-C","Section-C cell",       "fig:report_roc",     "Report RoC/PR curves"),
    ("fig_report_disorder_detection","Supp-C","Section-C cell",       "fig:disorder",       "Disorder detection metrics"),
    ("fig_report_pathology_dist",    "Supp-C","Section-C cell",       "fig:patho_dist",     "Pathology distribution"),
    ("fig_report_text_analysis",     "Supp-C","Section-C cell",       "fig:report_text",    "Report text analysis"),
    ("fig_bench_ablation_bars",      "Supp-D","Section-D cell",       "fig:abl_bars",       "Ablation bars (val)"),
    ("fig_bench_efficiency",         "Supp-D","Section-D cell",       "fig:efficiency",     "Efficiency scatter"),
    ("fig_bench_sam_baseline",       "Supp-D","Section-D cell",       "fig:sam_base",       "SAM baseline comparison"),
    ("fig_bench_bootstrap_ci",       "Supp-D","Section-D cell",       "fig:bootstrap",      "Bootstrap CI forest plot"),
    ("fig_bench_radar",              "Supp-D","Section-D cell",       "fig:radar",          "Multi-metric radar"),
    ("fig_bench_dataset_split",      "Supp-D","Section-D cell",       "fig:split",          "Dataset split pie/bar"),
    ("fig_bench_ablation_heatmap",   "Supp-D","Section-D cell",       "fig:abl_heat",       "Ablation metric heatmap"),
    ("fig_bench_statistical_significance","Supp-D+","Section-D+ cell","fig:stat_sig",       "Statistical significance plot"),
    ("fig_bench_test_baseline_context","Supp-D+","Section-D+ cell",  "fig:test_ctx",       "Test baseline context"),
    ("fig_review_prompt_gate",       "Review","Section-G cell",       "fig:gate",           "Prompt gate activation heatmap"),
    ("fig_review_feature_space_tsne","Review","Section-G cell",       "fig:tsne",           "Feature space t-SNE"),
]

df_manifest = pd.DataFrame(manifest,
    columns=["Figure Stem","Group","Source Script","TeX Label","Caption"])
display(df_manifest.to_string(index=False))

# Check which figures already have PNG/PDF in figures/
print('\n── Coverage check ──')
for stem, *_ in manifest:
    png = os.path.exists(os.path.join(FIGURES, f'{stem}.png'))
    pdf = os.path.exists(os.path.join(FIGURES, f'{stem}.pdf'))
    dis = os.path.exists(os.path.join(FIGURES_D, f'{stem}.png'))
    status = ('✓ PNG+PDF+DISP' if (png and pdf and dis)
              else '✓ PNG+PDF' if (png and pdf)
              else '✓ PNG+DISP' if (png and dis)
              else '✓ PNG' if png
              else '✗ MISSING')
    print(f"  {status:20s}  {stem}")


## Section A - Segmentation Figures

Figures characterising the prompted segmentation performance of PCRadNet (A2) across anatomical structures, imaging planes, disc levels, and ablation variants.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2 (Fig. A1): Per-Class Dice Heatmap - Axial & Sagittal
# ══════════════════════════════════════════════════════════════════════
# Shows Dice for each anatomical class, split by imaging plane, for A2.

axial_classes = ['dice_IVD', 'dice_PE', 'dice_SC', 'dice_VB']
axial_labels  = ['IVD', 'Post. Elements', 'Spinal Canal', 'Vert. Body']
sag_classes   = ['dice_IVD_sag', 'dice_Sacrum', 'dice_SC_sag']
sag_labels    = ['IVD (sag)', 'Sacrum', 'Spinal Canal (sag)']

a2 = [m for m in abl_metrics if m['ablation'] == 'A2'][0]

# Build matrix [2 rows (axial/sagittal) × 7 cols (classes)]
all_labels = axial_labels + sag_labels
dice_axial = [a2[c] for c in axial_classes] + [np.nan]*3
dice_sag   = [np.nan]*4 + [a2[c] for c in sag_classes]
mat = np.array([dice_axial, dice_sag])

fig, ax = plt.subplots(figsize=(5.5, 1.8))
mask = np.isnan(mat)
sns.heatmap(mat, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1,
            mask=mask, xticklabels=all_labels, yticklabels=['Axial', 'Sagittal'],
            linewidths=0.8, linecolor='white', cbar_kws={'label': 'Dice', 'shrink': 0.8},
            annot_kws={'fontsize': 8.5, 'fontweight': 'bold'}, ax=ax)
ax.set_title('Per-Class Dice Scores (A2, Test Set)', fontsize=10, pad=8)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
save_fig(fig, 'fig_seg_perclass_heatmap')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3 (Fig. A2): Prompted Dice Distribution - Axial vs Sagittal
# ══════════════════════════════════════════════════════════════════════
# Split violin + strip showing the per-image Dice distribution by plane.

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8),
                         gridspec_kw={'width_ratios': [1, 1.2]})

# (a) Violin plot
parts = axes[0].violinplot(
    [df_a2.loc[df_a2.plane == 'axial', 'prompted_dice'].values,
     df_a2.loc[df_a2.plane == 'sagittal', 'prompted_dice'].values],
    positions=[0, 1], showmeans=True, showmedians=True, showextrema=False)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor([PAL['blue'], PAL['orange']][i])
    pc.set_alpha(0.55)
parts['cmeans'].set_color('k')
parts['cmedians'].set_color(PAL['red'])
parts['cmedians'].set_linestyle('--')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Axial', 'Sagittal'])
axes[0].set_ylabel('Prompted Dice')
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title('(a) Dice by Imaging Plane')
# add n=
for i, plane in enumerate(['axial', 'sagittal']):
    n = (df_a2.plane == plane).sum()
    m = df_a2.loc[df_a2.plane == plane, 'prompted_dice'].mean()
    axes[0].text(i, -0.03, f'n={n}\nμ={m:.3f}', ha='center', va='top', fontsize=7)

# (b) Histogram overlay
for plane, col in [('axial', PAL['blue']), ('sagittal', PAL['orange'])]:
    vals = df_a2.loc[df_a2.plane == plane, 'prompted_dice']
    axes[1].hist(vals, bins=40, alpha=0.55, color=col, edgecolor='k',
                 linewidth=0.3, label=f'{plane.capitalize()} (n={len(vals)})', density=True)
axes[1].set_xlabel('Prompted Dice')
axes[1].set_ylabel('Density')
axes[1].set_title('(b) Dice Density by Plane')
axes[1].legend(fontsize=7, framealpha=0.9)

plt.tight_layout()
save_fig(fig, 'fig_seg_axial_vs_sagittal')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4 (Fig. A3): Per-Disc-Level Prompted Dice & IoU (Axial)
# ══════════════════════════════════════════════════════════════════════
axial = df_a2[df_a2.plane == 'axial'].copy()
level_stats = axial.groupby('level').agg(
    dice_mean=('prompted_dice', 'mean'),
    dice_std=('prompted_dice', 'std'),
    iou_mean=('prompted_iou', 'mean'),
    iou_std=('prompted_iou', 'std'),
    n=('prompted_dice', 'count')
).reindex(['D3', 'D4', 'D5'])

fig, ax = plt.subplots(figsize=(3.8, 2.8))
x = np.arange(3)
w = 0.32

bars1 = ax.bar(x - w/2, level_stats['dice_mean'], w,
               yerr=level_stats['dice_std'], capsize=3,
               color=PAL['blue'], edgecolor='k', linewidth=0.4, label='Dice')
bars2 = ax.bar(x + w/2, level_stats['iou_mean'], w,
               yerr=level_stats['iou_std'], capsize=3,
               color=PAL['orange'], edgecolor='k', linewidth=0.4, label='IoU')

for i, (d, j, n) in enumerate(zip(level_stats['dice_mean'],
                                    level_stats['iou_mean'],
                                    level_stats['n'])):
    ax.text(i, max(d, j) + level_stats['dice_std'].iloc[i] + 0.04,
            f'n={int(n)}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(['L3/L4', 'L4/L5', 'L5/S1'])
ax.set_xlabel('Disc Level')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.set_title('Per-Level Dice & IoU (Axial, A2 Test)')
ax.legend(fontsize=7, loc='lower right')
plt.tight_layout()
save_fig(fig, 'fig_seg_per_disc_level')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 5 (Fig. A4): Pixel-Level Class Imbalance
# ══════════════════════════════════════════════════════════════════════
# Visualises the severe class imbalance in both axial and sagittal views.

ax_counts = class_wts['axial_pixel_counts']
sg_counts = class_wts['sagittal_pixel_counts']

axial_cls_names  = ['Background', 'IVD', 'Post. Elements', 'Spinal Canal']
sag_cls_names    = ['Background', 'IVD', 'Spinal Canal', 'Sacrum', 'Vert. Body']

fig, axes = plt.subplots(1, 2, figsize=(7, 2.6))

# (a) Axial
vals_ax = [ax_counts[str(i)] for i in range(4)]
log_vals = np.log10(np.array(vals_ax))
bars = axes[0].barh(axial_cls_names, log_vals, color=[PAL['grey'], PAL['blue'],
                    PAL['green'], PAL['red']], edgecolor='k', linewidth=0.4)
for b, v in zip(bars, vals_ax):
    axes[0].text(b.get_width() + 0.05, b.get_y() + b.get_height()/2,
                 f'{v:,.0f}', va='center', fontsize=7)
axes[0].set_xlabel('log₁₀(Pixel Count)')
axes[0].set_title('(a) Axial Class Distribution')

# (b) Sagittal
vals_sg = [sg_counts[str(i)] for i in range(5)]
log_vals_s = np.log10(np.array(vals_sg))
bars = axes[1].barh(sag_cls_names, log_vals_s,
                    color=[PAL['grey'], PAL['blue'], PAL['red'],
                           PAL['orange'], PAL['teal']],
                    edgecolor='k', linewidth=0.4)
for b, v in zip(bars, vals_sg):
    axes[1].text(b.get_width() + 0.05, b.get_y() + b.get_height()/2,
                 f'{v:,.0f}', va='center', fontsize=7)
axes[1].set_xlabel('log₁₀(Pixel Count)')
axes[1].set_title('(b) Sagittal Class Distribution')

plt.tight_layout()
save_fig(fig, 'fig_seg_class_imbalance')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 6 (Fig. A5): Dice Score Distribution + Failure Analysis
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

# (a) Histogram with failure threshold
dice_vals = df_a2['prompted_dice'].values
fail_pct = (dice_vals < 0.5).mean() * 100
axes[0].hist(dice_vals, bins=50, color=PAL['blue'], edgecolor='k',
             linewidth=0.3, alpha=0.8)
axes[0].axvline(0.5, color=PAL['red'], ls='--', lw=1.0,
                label=f'Failure threshold (Dice < 0.5): {fail_pct:.1f}%')
axes[0].axvline(np.median(dice_vals), color='k', ls='-', lw=0.8,
                label=f'Median = {np.median(dice_vals):.3f}')
axes[0].set_xlabel('Prompted Dice Score')
axes[0].set_ylabel('Count')
axes[0].set_title(f'(a) Dice Distribution (A2 Test, n={len(dice_vals)})')
axes[0].legend(fontsize=6.5)

# (b) Boxplot per prompt type (top 8)
top_prompts = df_a2['prompt'].value_counts().head(8).index.tolist()
data_bp = [df_a2[df_a2.prompt == p]['prompted_dice'].values for p in top_prompts]
bp = axes[1].boxplot(data_bp, vert=True, patch_artist=True, widths=0.55,
                     medianprops=dict(color='k', linewidth=1.2),
                     flierprops=dict(markersize=1.5, alpha=0.4))
for patch in bp['boxes']:
    patch.set_facecolor(PAL['blue'])
    patch.set_alpha(0.5)
labels_short = [p[:14] for p in top_prompts]
axes[1].set_xticklabels(labels_short, rotation=40, ha='right', fontsize=6.5)
axes[1].set_ylabel('Prompted Dice')
axes[1].set_title('(b) Dice by Prompt Type')

plt.tight_layout()
save_fig(fig, 'fig_seg_dice_distribution')
plt.show()
display(fig)

# Print failure cases
worst = df_a2.nsmallest(5, 'prompted_dice')[
    ['fname', 'level', 'plane', 'prompt', 'prompted_dice']]
print('Worst 5 images:')
print(worst.to_string(index=False))

## Section B - Severity Grading Figures

Pfirrmann grading analysis: confusion matrices, per-grade metrics, ordinal calibration, and grade error distribution.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 7 (Fig. B1): Normalised Confusion Matrix + Class Imbalance
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(7, 3.0),
                         gridspec_kw={'width_ratios': [1.3, 0.7]})

# (a) Normalised confusion matrix
cm_norm = cm_raw / cm_raw.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)
grade_labels = ['I', 'II', 'III', 'IV', 'V']

im = axes[0].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1, aspect='equal')
for i in range(5):
    for j in range(5):
        val = cm_norm[i, j]
        count = int(cm_raw[i, j])
        color = 'white' if val > 0.5 else 'black'
        axes[0].text(j, i, f'{val:.2f}\n({count})', ha='center', va='center',
                     fontsize=7, color=color, fontweight='bold' if i == j else 'normal')
axes[0].set_xticks(range(5))
axes[0].set_yticks(range(5))
axes[0].set_xticklabels(grade_labels)
axes[0].set_yticklabels(grade_labels)
axes[0].set_xlabel('Predicted Grade')
axes[0].set_ylabel('True Grade')
axes[0].set_title('(a) Normalised Confusion Matrix (A2)')
plt.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label='Proportion')

# (b) Class distribution bars
supports = cm_raw.sum(axis=1).astype(int)
axes[1].barh(range(5), supports, color=GRADE_COLORS,
             edgecolor='k', linewidth=0.4)
axes[1].set_yticks(range(5))
axes[1].set_yticklabels(grade_labels)
axes[1].set_xlabel('Sample Count')
axes[1].set_title('(b) Test Set Distribution')
for i, s in enumerate(supports):
    axes[1].text(s + 2, i, str(s), va='center', fontsize=7)
axes[1].invert_yaxis()

plt.tight_layout()
save_fig(fig, 'fig_grade_confusion_matrix')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 8 (Fig. B2): Per-Grade Precision, Recall, F1 + Support
# ══════════════════════════════════════════════════════════════════════
grades = ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4', 'Grade 5']
prec = [cls_report[g]['precision'] for g in grades]
rec  = [cls_report[g]['recall'] for g in grades]
f1s  = [cls_report[g]['f1-score'] for g in grades]
sups = [int(cls_report[g]['support']) for g in grades]

fig, ax1 = plt.subplots(figsize=(4.5, 3.0))
x = np.arange(5)
w = 0.22

ax1.bar(x - w, prec, w, color=PAL['blue'], edgecolor='k', linewidth=0.4,
        label='Precision')
ax1.bar(x, rec, w, color=PAL['orange'], edgecolor='k', linewidth=0.4,
        label='Recall')
ax1.bar(x + w, f1s, w, color=PAL['green'], edgecolor='k', linewidth=0.4,
        label='F1-score')
ax1.set_xticks(x)
ax1.set_xticklabels(['I', 'II', 'III', 'IV', 'V'])
ax1.set_xlabel('Pfirrmann Grade')
ax1.set_ylabel('Score')
ax1.set_ylim(0, 1.05)
ax1.set_title('Per-Grade Classification Metrics (A2, Test)')

# Overlay support text
for i, s in enumerate(sups):
    ax1.text(i, max(prec[i], rec[i], f1s[i]) + 0.04, f'n={s}',
             ha='center', va='bottom', fontsize=6.5, color=PAL['grey'])

ax1.legend(fontsize=7, loc='upper right', ncol=3)
plt.tight_layout()
save_fig(fig, 'fig_grade_per_grade_metrics')
plt.show()
display(fig)

print(f'Macro F1: {cls_report["macro avg"]["f1-score"]:.3f}')
print(f'Weighted F1: {cls_report["weighted avg"]["f1-score"]:.3f}')
print(f'Accuracy: {cls_report["accuracy"]:.3f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 9 (Fig. B3): Ordinal Grade Error Distribution
# ══════════════════════════════════════════════════════════════════════
# Shows how far predictions deviate from ground truth (should cluster at 0).

graded = df_a2[df_a2['gt_grade'].isin([1,2,3,4,5])].copy()
grade_error = graded['pred_grade'] - graded['gt_grade']

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

# (a) Error histogram
error_counts = grade_error.value_counts().sort_index()
all_errors = range(int(grade_error.min()), int(grade_error.max()) + 1)
counts_full = [error_counts.get(e, 0) for e in all_errors]
colors_e = [PAL['green'] if e == 0 else (PAL['blue'] if abs(e) == 1 else PAL['red'])
            for e in all_errors]
axes[0].bar(all_errors, counts_full, color=colors_e, edgecolor='k', linewidth=0.4)
for e, c in zip(all_errors, counts_full):
    if c > 0:
        axes[0].text(e, c + 3, str(c), ha='center', va='bottom', fontsize=6.5)
axes[0].set_xlabel('Prediction Error (Predicted − True)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'(a) Ordinal Error Distribution (n={len(graded)})')

within1 = (abs(grade_error) <= 1).mean() * 100
exact = (grade_error == 0).mean() * 100
axes[0].text(0.98, 0.95, f'Exact: {exact:.1f}%\nWithin ±1: {within1:.1f}%',
             transform=axes[0].transAxes, ha='right', va='top', fontsize=7,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

# (b) Per-grade error boxplots
data_by_grade = [graded.loc[graded.gt_grade == g, 'grade_error'].values
                 for g in [1, 2, 3, 4, 5]]
bp = axes[1].boxplot(data_by_grade, vert=True, patch_artist=True,
                     medianprops=dict(color='k', linewidth=1.2),
                     flierprops=dict(markersize=2, alpha=0.5), widths=0.55)
for patch, c in zip(bp['boxes'], GRADE_COLORS):
    patch.set_facecolor(c)
    patch.set_alpha(0.6)
axes[1].set_xticklabels(['I', 'II', 'III', 'IV', 'V'])
axes[1].set_xlabel('True Pfirrmann Grade')
axes[1].set_ylabel('Prediction Error')
axes[1].axhline(0, color='k', ls='--', lw=0.5, alpha=0.5)
axes[1].set_title('(b) Error by True Grade')

plt.tight_layout()
save_fig(fig, 'fig_grade_error_distribution')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 10 (Fig. B4): Bland-Altman Plot - Disc Height Estimation (mm)
# ══════════════════════════════════════════════════════════════════════
# A6: converted pixel heights to mm using MEAN_SPACING_MM = 0.49
MEAN_SPACING_MM = 0.49

ht_df = df_a2[(df_a2['gt_height'] > 0) & (df_a2['pred_height'] > 0)].copy()
gt_ht_mm   = ht_df['gt_height'] * MEAN_SPACING_MM
pred_ht_mm = ht_df['pred_height'] * MEAN_SPACING_MM
mean_ht = (gt_ht_mm + pred_ht_mm) / 2
diff_ht = pred_ht_mm - gt_ht_mm
bias = diff_ht.mean()
sd   = diff_ht.std()
loa_hi = bias + 1.96 * sd
loa_lo = bias - 1.96 * sd
r_val  = gt_ht_mm.corr(pred_ht_mm)

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

# (a) Bland-Altman
axes[0].scatter(mean_ht, diff_ht, s=4, alpha=0.3, color=PAL['blue'],
                edgecolors='none', rasterized=True)
axes[0].axhline(bias, color='k', ls='-', lw=0.8,
                label=f'Bias = {bias:+.2f} mm')
axes[0].axhline(loa_hi, color=PAL['red'], ls='--', lw=0.8,
                label=f'+1.96 SD = {loa_hi:+.2f} mm')
axes[0].axhline(loa_lo, color=PAL['red'], ls='--', lw=0.8,
                label=f'−1.96 SD = {loa_lo:+.2f} mm')
axes[0].fill_between(axes[0].get_xlim(), loa_lo, loa_hi, alpha=0.06, color=PAL['red'])
axes[0].set_xlabel('Mean Height (mm)')
axes[0].set_ylabel('Predicted − GT (mm)')
axes[0].set_title(f'(a) Bland–Altman (n={len(ht_df)})')
axes[0].legend(fontsize=6, loc='upper left', framealpha=0.9)

# (b) Scatter: GT vs Predicted
axes[1].scatter(gt_ht_mm, pred_ht_mm, s=4, alpha=0.3,
                color=PAL['green'], edgecolors='none', rasterized=True)
lims = [min(gt_ht_mm.min(), pred_ht_mm.min()) - 0.5,
        max(gt_ht_mm.max(), pred_ht_mm.max()) + 0.5]
axes[1].plot(lims, lims, 'k--', lw=0.8, alpha=0.5, label='y = x')
axes[1].set_xlabel('Ground Truth Height (mm)')
axes[1].set_ylabel('Predicted Height (mm)')
axes[1].set_title(f'(b) GT vs Predicted (r = {r_val:.3f})')
axes[1].legend(fontsize=7)
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_aspect('equal')

plt.tight_layout()
save_fig(fig, 'fig_grade_bland_altman')
plt.show()
display(fig)
print(f'Disc height — MAE: {diff_ht.abs().mean():.3f} mm, Bias: {bias:.3f} mm, '
      f'LoA: [{loa_lo:.2f}, {loa_hi:.2f}] mm, r: {r_val:.3f}')

## Section C - Radiological Reporting Figures

Disorder detection, report structure analysis, and comparison with reference structured findings.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 11 (Fig. C1): ROC Curves - Disorder Detection (A2)
# ══════════════════════════════════════════════════════════════════════
disorders = [
    ('Bulge',       'disorder_Bulge_prob',       'disorder_Bulge_gt'),
    ('Thecal Comp.','disorder_ThecalComp_prob',  'disorder_ThecalComp_gt'),
    ('Nerve Comp.', 'disorder_NerveComp_prob',   'disorder_NerveComp_gt'),
    ('Protrusion',  'disorder_Protrusion_prob',  'disorder_Protrusion_gt'),
    ('Herniation',  'disorder_Herniation_prob',  'disorder_Herniation_gt'),
    ('Stenosis',    'disorder_Stenosis_prob',    'disorder_Stenosis_gt'),
]
roc_colors = [PAL['blue'], PAL['green'], PAL['orange'],
              PAL['red'], PAL['purple'], PAL['teal']]

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))

# (a) ROC curves
for (name, pcol, gcol), c in zip(disorders, roc_colors):
    mask = df_a2[pcol].notna() & df_a2[gcol].notna()
    y_true = df_a2.loc[mask, gcol].values
    y_score = df_a2.loc[mask, pcol].values
    if y_true.sum() > 0 and (1 - y_true).sum() > 0:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        axes[0].plot(fpr, tpr, color=c, lw=1.0,
                     label=f'{name} ({roc_auc:.2f})')
axes[0].plot([0,1], [0,1], 'k--', lw=0.5, alpha=0.4)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('(a) ROC Curves')
axes[0].legend(fontsize=6, loc='lower right', framealpha=0.9, title='AUC', title_fontsize=6)
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])
axes[0].set_aspect('equal')

# (b) Precision-Recall curves
for (name, pcol, gcol), c in zip(disorders, roc_colors):
    mask = df_a2[pcol].notna() & df_a2[gcol].notna()
    y_true = df_a2.loc[mask, gcol].values
    y_score = df_a2.loc[mask, pcol].values
    if y_true.sum() > 0:
        prec_arr, rec_arr, _ = precision_recall_curve(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        axes[1].plot(rec_arr, prec_arr, color=c, lw=1.0,
                     label=f'{name} ({ap:.2f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('(b) Precision-Recall Curves')
axes[1].legend(fontsize=6, loc='lower left', framealpha=0.9, title='AP', title_fontsize=6)
axes[1].set_xlim([-0.02, 1.02])
axes[1].set_ylim([-0.02, 1.05])

plt.tight_layout()
save_fig(fig, 'fig_report_roc_pr_curves')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 12 (Fig. C2): Disorder Detection - Balanced Accuracy / Precision / Recall
# ══════════════════════════════════════════════════════════════════════
# A5: replaced raw accuracy with balanced accuracy computed from per-image data.
from sklearn.metrics import balanced_accuracy_score

a2m = [m for m in abl_metrics if m['ablation'] == 'A2'][0]
disorder_names_full = [
    'Bulge', 'ThecalComp', 'NerveComp', 'Protrusion', 'Herniation',
    'Stenosis', 'Desiccated', 'LigFlavum', 'Spondylolisthesis'
]
display_names = [
    'Bulge', 'Thecal\nComp.', 'Nerve\nComp.', 'Protrusion', 'Herniation',
    'Stenosis', 'Desiccated', 'Lig.\nFlavum', 'Spondylo-\nlisthesis'
]

# Compute balanced accuracy from per-image predictions in df_a2
baccs = []
for d in disorder_names_full:
    gt_col = f'disorder_{d}_gt'
    prob_col = f'disorder_{d}_prob'
    if gt_col in df_a2.columns and prob_col in df_a2.columns:
        mask = df_a2[gt_col].notna() & df_a2[prob_col].notna()
        y_true = df_a2.loc[mask, gt_col].values.astype(int)
        y_pred = (df_a2.loc[mask, prob_col].values >= 0.5).astype(int)
        if len(y_true) > 0 and len(set(y_true)) > 1:
            baccs.append(balanced_accuracy_score(y_true, y_pred))
        else:
            baccs.append(0.0)
    else:
        baccs.append(a2m.get(f'disorder_{d}_acc', 0.0))

precs = [a2m[f'disorder_{d}_prec'] for d in disorder_names_full]
recs  = [a2m[f'disorder_{d}_rec'] for d in disorder_names_full]

fig, ax = plt.subplots(figsize=(7, 3.0))
x = np.arange(len(disorder_names_full))
w = 0.25

ax.bar(x - w, baccs, w, color=PAL['blue'], edgecolor='k', linewidth=0.4, label='Bal. Accuracy')
ax.bar(x, precs, w, color=PAL['orange'], edgecolor='k', linewidth=0.4, label='Precision')
ax.bar(x + w, recs, w, color=PAL['green'], edgecolor='k', linewidth=0.4, label='Recall')

ax.set_xticks(x)
ax.set_xticklabels(display_names, fontsize=7)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.08)
ax.set_title('Disorder Detection Performance (A2, Test Set)')
ax.legend(fontsize=7, loc='upper right', ncol=3)

# Highlight disorders with zero precision/recall
for i, (p, r) in enumerate(zip(precs, recs)):
    if p == 0 and r == 0:
        ax.text(i, 0.02, '—', ha='center', va='bottom', fontsize=8, color=PAL['red'])

plt.tight_layout()
save_fig(fig, 'fig_report_disorder_detection')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 13 (Fig. C3): Reference Pathology Distribution
# ══════════════════════════════════════════════════════════════════════
# Frequency of pathologies in the reference structured reports.

path_counts = ref_reports['Pathology'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(5.5, 3.2))
bars = ax.barh(range(len(path_counts)), path_counts.values[::-1],
               color=PAL['blue'], edgecolor='k', linewidth=0.4, alpha=0.8)
ax.set_yticks(range(len(path_counts)))
ax.set_yticklabels(path_counts.index[::-1], fontsize=7.5)
for i, v in enumerate(path_counts.values[::-1]):
    ax.text(v + 2, i, str(v), va='center', fontsize=7)
ax.set_xlabel('Finding Count')
ax.set_title(f'Reference Pathology Distribution ({ref_reports.Patient_ID.nunique()} patients, '
             f'{len(ref_reports)} findings)')
plt.tight_layout()
save_fig(fig, 'fig_report_pathology_distribution')
plt.show()
display(fig)

# Negated findings
neg_pct = ref_reports['Is_Negated'].mean() * 100
print(f'Negated findings: {neg_pct:.1f}% of all reference findings')
print(f'Top levels: {ref_reports["Level"].value_counts().head(6).to_dict()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 15 (Fig. C4): Generated Report Example + Segmentation Confidence
# ══════════════════════════════════════════════════════════════════════
report_dir = os.path.join(RESULTS, 'reports')
sample_path = os.path.join(report_dir, 'report_0003.txt')

# Read with fallback encoding
with open(sample_path, encoding='utf-8', errors='replace') as f:
    report_text = f.read()
print('=' * 60)
print('SAMPLE GENERATED REPORT (Patient ID: 3)')
print('=' * 60)
print(report_text)

# Show per-level metrics for a test patient
pat4 = df_a2[df_a2.patient_id == 4]
if len(pat4) > 0:
    print('\n' + '=' * 60)
    print('PER-IMAGE METRICS (Patient 4, test set)')
    print('=' * 60)
    for _, row in pat4.iterrows():
        dice = row['prompted_dice']
        conf = 'good' if dice > 0.8 else ('moderate' if dice > 0.65 else 'limited')
        print(f"  {row['fname']}: Dice={dice:.3f} ({conf}), "
              f"Grade={int(row.get('pred_grade',0))}, "
              f"Height={row.get('pred_height',0):.1f}px")

# Reference findings for Patient 3
ref3 = ref_reports[ref_reports['Patient_ID'] == 3]
if len(ref3) > 0:
    print('\n' + '=' * 60)
    print('REFERENCE FINDINGS (Patient 3)')
    print('=' * 60)
    for _, row in ref3.iterrows():
        seg = str(row.get('Original_Segment', '?'))[:80]
        print(f"  Level: {row.get('Level','?')}, Pathology: {row.get('Pathology','?')}, "
              f"Details: {seg}")

## Section D - Benchmark & Ablation Figures

Ablation study comparison, computational efficiency, statistical significance, SAM baseline comparison, bootstrap confidence intervals.

In [ ]:
# ══════════════════════════════════════════════════════════════════════

# Cell 16 (Fig. D1): Ablation Study - Multi-Panel Bar Chart (Test)

# ══════════════════════════════════════════════════════════════════════

variants = ['A0', 'A1', 'A2', 'A3']

abl_map = {m['ablation']: m for m in abl_metrics}



m_dice = [abl_map[v]['prompted_dice_mean'] for v in variants]

s_dice = [abl_map[v]['prompted_dice_std'] for v in variants]

m_iou  = [abl_map[v]['prompted_iou_mean'] for v in variants]

m_gacc = [abl_map[v]['grade_accuracy'] for v in variants]

m_w1   = [abl_map[v]['grade_within1'] for v in variants]



fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.6))

x = np.arange(4)

w = 0.55

colors = [ABL_COLORS[v] for v in variants]



panels = [

    (axes[0], m_dice, s_dice, 'Prompted Dice', '(a)'),

    (axes[1], m_iou, None, 'Prompted IoU', '(b)'),

    (axes[2], m_gacc, None, 'Grade Accuracy', '(c)'),

    (axes[3], m_w1, None, 'Within ±1', '(d)'),

]



for ax, vals, errs, ylabel, label in panels:

    kw = dict(yerr=errs, capsize=3) if errs else {}

    ax.bar(x, vals, w, color=colors, edgecolor='k', linewidth=0.4, **kw)

    ax.set_ylabel(ylabel)

    ax.set_title(label)

    ax.set_ylim(0, 1.12)

    ax.set_xticks(x)

    ax.set_xticklabels(variants, fontsize=8)

    for i, v in enumerate(vals):

        y_off = (errs[i] if errs else 0) + 0.03

        ax.text(i, v + y_off, f'{v:.3f}', ha='center', va='bottom', fontsize=6.5)



plt.tight_layout()

save_fig(fig, 'fig_bench_ablation_bars')

plt.show()

display(fig)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 16 (Fig. D2): Computational Efficiency - Params vs Performance
# ══════════════════════════════════════════════════════════════════════
params = [m['trainable_params'] / 1e6 for m in abl_metrics]  # in millions
vrams  = [m['vram_gb'] for m in abl_metrics]
dices  = [m['prompted_dice_mean'] for m in abl_metrics]
total_params_M = phase2_info['total_params'] / 1e6  # total model

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

# (a) Trainable params vs Dice
for i, v in enumerate(variants):
    axes[0].scatter(params[i], dices[i], s=80, color=ABL_COLORS[v],
                    edgecolors='k', linewidth=0.6, zorder=3)
    axes[0].annotate(v, (params[i], dices[i]),
                     textcoords='offset points', xytext=(6, 4), fontsize=8)
axes[0].set_xlabel('Trainable Parameters (M)')
axes[0].set_ylabel('Prompted Dice')
axes[0].set_title('(a) Parameters vs Segmentation')
axes[0].axhline(dices[2], color=PAL['green'], ls=':', lw=0.6, alpha=0.5)
# Add total params annotation
axes[0].text(0.02, 0.02, f'Total: {total_params_M:.1f}M\n'
             f'Trainable: {params[2]:.2f}M ({params[2]/total_params_M*100:.1f}%)',
             transform=axes[0].transAxes, fontsize=6.5, va='bottom',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# (b) VRAM vs Dice
for i, v in enumerate(variants):
    axes[1].scatter(vrams[i], dices[i], s=80, color=ABL_COLORS[v],
                    edgecolors='k', linewidth=0.6, zorder=3)
    axes[1].annotate(v, (vrams[i], dices[i]),
                     textcoords='offset points', xytext=(6, 4), fontsize=8)
axes[1].set_xlabel('Peak VRAM (GB)')
axes[1].set_ylabel('Prompted Dice')
axes[1].set_title('(b) VRAM vs Segmentation')

plt.tight_layout()
save_fig(fig, 'fig_bench_efficiency')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 17 (Fig. D3): SAM Baseline - Per-Class Comparison (Test)
# ══════════════════════════════════════════════════════════════════════
# Compares zero-shot SAM ViT-B box-prompted vs PCRadNet per class.
# A4: switched from validation to test data.

# SAM baseline per class from TEST CSV
sam_by_cls = sam_test_df.groupby('target_cls')['dice'].mean()
axial_sam = sam_test_df[sam_test_df.plane == 'axial'].groupby('target_cls')['dice'].mean()
sag_sam   = sam_test_df[sam_test_df.plane == 'sagittal'].groupby('target_cls')['dice'].mean()

# PCRadNet A2 TEST metrics
a2_test = [m for m in abl_metrics if m['ablation'] == 'A2'][0]

# Axial classes: IVD(1), PE(2), SC(3)
ax_cls_names = ['IVD', 'Post. Elements', 'Spinal Canal']
sam_ax = [axial_sam.get(i, 0) for i in [1, 2, 3]]
prs_ax = [a2_test['dice_IVD'], a2_test['dice_PE'], a2_test['dice_SC']]

# Sagittal classes: IVD_sag(1), SC_sag(2), Sacrum(3), VB(4)
sg_cls_names = ['IVD (sag)', 'Spinal Canal (sag)', 'Sacrum', 'Vert. Body']
sam_sg = [sag_sam.get(i, 0) for i in [1, 2, 3, 4]]
prs_sg = [a2_test['dice_IVD_sag'], a2_test['dice_SC_sag'],
          a2_test['dice_Sacrum'], a2_test.get('dice_VB', 0)]

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

# (a) Axial
x_a = np.arange(len(ax_cls_names))
axes[0].bar(x_a - 0.18, sam_ax, 0.32, color=PAL['grey'], edgecolor='k',
            linewidth=0.4, label='SAM ViT-B')
axes[0].bar(x_a + 0.18, prs_ax, 0.32, color=PAL['green'], edgecolor='k',
            linewidth=0.4, label='PCRadNet')
axes[0].set_xticks(x_a)
axes[0].set_xticklabels(ax_cls_names, fontsize=7.5)
axes[0].set_ylabel('Dice')
axes[0].set_ylim(0, 1.12)
axes[0].set_title('(a) Axial — Per-Class Dice (Test)')
axes[0].legend(fontsize=6.5)
for i in range(len(ax_cls_names)):
    axes[0].text(i - 0.18, sam_ax[i] + 0.02, f'{sam_ax[i]:.2f}',
                 ha='center', fontsize=6)
    axes[0].text(i + 0.18, prs_ax[i] + 0.02, f'{prs_ax[i]:.2f}',
                 ha='center', fontsize=6)

# (b) Sagittal
x_s = np.arange(len(sg_cls_names))
axes[1].bar(x_s - 0.18, sam_sg, 0.32, color=PAL['grey'], edgecolor='k',
            linewidth=0.4, label='SAM ViT-B')
axes[1].bar(x_s + 0.18, prs_sg, 0.32, color=PAL['green'], edgecolor='k',
            linewidth=0.4, label='PCRadNet')
axes[1].set_xticks(x_s)
axes[1].set_xticklabels(sg_cls_names, fontsize=7, rotation=15, ha='right')
axes[1].set_ylabel('Dice')
axes[1].set_ylim(0, 1.12)
axes[1].set_title('(b) Sagittal — Per-Class Dice (Test)')
axes[1].legend(fontsize=6.5)
for i in range(len(sg_cls_names)):
    axes[1].text(i - 0.18, sam_sg[i] + 0.02, f'{sam_sg[i]:.2f}',
                 ha='center', fontsize=6)
    axes[1].text(i + 0.18, prs_sg[i] + 0.02, f'{prs_sg[i]:.2f}',
                 ha='center', fontsize=6)

plt.tight_layout()
save_fig(fig, 'fig_bench_sam_baseline_comparison')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 18 (Fig. D4): Bootstrap 95% CIs (from preloaded data)
# ══════════════════════════════════════════════════════════════════════
# A10: Source from preloaded bootstrap_cis instead of recomputing.

ci_results = {}
for abl_key in ['A0', 'A1', 'A2', 'A3']:
    abl_ci = bootstrap_cis.get(abl_key, {})
    ci_results[abl_key] = {
        'dice': (abl_ci.get('mean_dice', 0),
                 abl_ci.get('ci_dice_lo', 0),
                 abl_ci.get('ci_dice_hi', 0)),
        'iou':  (abl_ci.get('mean_iou', 0),
                 abl_ci.get('ci_iou_lo', 0),
                 abl_ci.get('ci_iou_hi', 0)),
    }
    m_d, lo_d, hi_d = ci_results[abl_key]['dice']
    m_j, lo_j, hi_j = ci_results[abl_key]['iou']
    print(f'{abl_key}: Dice = {m_d:.3f} [{lo_d:.3f}, {hi_d:.3f}], '
          f'IoU = {m_j:.3f} [{lo_j:.3f}, {hi_j:.3f}]')

# ── Figure: CI forest plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(7, 2.4))
abl_names = ['A0', 'A1', 'A2', 'A3']
y_pos = np.arange(4)

for idx, (metric, title) in enumerate([('dice', 'Prompted Dice'),
                                        ('iou', 'Prompted IoU')]):
    ax = axes[idx]
    for i, abl in enumerate(abl_names):
        m, lo, hi = ci_results[abl][metric]
        ax.errorbar(m, i, xerr=[[m - lo], [hi - m]], fmt='o', color=ABL_COLORS[abl],
                    markersize=6, capsize=4, capthick=1.2, elinewidth=1.2)
        ax.text(hi + 0.008, i, f'{m:.3f} [{lo:.3f}, {hi:.3f}]',
                va='center', fontsize=6.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(abl_names)
    ax.set_xlabel(title)
    ax.set_title(f'95% Bootstrap CI — {title}')
    ax.invert_yaxis()

plt.tight_layout()
save_fig(fig, 'fig_bench_bootstrap_ci')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 19 (Fig. D5): Radar Chart - Multi-Task Ablation Comparison
# ══════════════════════════════════════════════════════════════════════
# A7: replaced max(r,0) with (r+1)/2 normalization for Height r.
categories = ['Dice', 'IoU', 'FG Dice', 'Grade Acc', 'Within ±1', 'Height r (norm)']
N_cats = len(categories)
angles = np.linspace(0, 2 * np.pi, N_cats, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(3.5, 3.5), subplot_kw=dict(polar=True))

for m_abl in abl_metrics:
    abl = m_abl['ablation']
    vals = [
        m_abl['prompted_dice_mean'],
        m_abl['prompted_iou_mean'],
        m_abl['mean_fg_dice'],
        m_abl['grade_accuracy'],
        m_abl['grade_within1'],
        (m_abl['height_pearson_r'] + 1) / 2,  # A7: (r+1)/2 normalization
    ]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', markersize=3, linewidth=1.0,
            color=ABL_COLORS[abl], label=abl)
    ax.fill(angles, vals, alpha=0.08, color=ABL_COLORS[abl])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=7)
ax.set_ylim(0, 1.0)
ax.set_yticks([0.2, 0.4, 0.6, 0.8])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8'], fontsize=6)
ax.set_title('Multi-Task Performance (Test Set)', pad=15, fontsize=10)
ax.legend(loc='lower right', bbox_to_anchor=(1.25, -0.05), fontsize=7)
plt.tight_layout()
save_fig(fig, 'fig_bench_radar')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 20 (Fig. D6): Dataset Split & Patient Distribution
# ══════════════════════════════════════════════════════════════════════
# Split composition, grade distribution per split, and imaging summary.

fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))

# (a) Split sizes (patients)
split_counts = split_df['Split'].value_counts()
order = ['train', 'val', 'test']
split_vals = [split_counts.get(s, 0) for s in order]
bars = axes[0].bar(order, split_vals,
                   color=[PAL['blue'], PAL['orange'], PAL['green']],
                   edgecolor='k', linewidth=0.4)
for b, v in zip(bars, split_vals):
    axes[0].text(b.get_x() + b.get_width()/2, v + 3, str(v),
                 ha='center', va='bottom', fontsize=7)
axes[0].set_ylabel('Number of Patients')
axes[0].set_title('(a) Patient Split')

# (b) Grade distribution per split (stacked)
grades_all = [1, 2, 3, 4, 5]
# From the split report, Max_Pfirrmann gives the patient max grade
for i, sp in enumerate(order):
    sp_data = split_df[split_df.Split == sp]['Max_Pfirrmann']
    g_counts = sp_data.value_counts().sort_index()
    bottom = 0
    for g in grades_all:
        cnt = g_counts.get(g, 0)
        axes[1].bar(i, cnt, bottom=bottom, color=GRADE_COLORS[g-1],
                    edgecolor='k', linewidth=0.3)
        if cnt > 10:
            axes[1].text(i, bottom + cnt/2, str(cnt), ha='center',
                         va='center', fontsize=6, color='white')
        bottom += cnt
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(['Train', 'Val', 'Test'])
axes[1].set_ylabel('Patient Count')
axes[1].set_title('(b) Max Grade per Split')
# Legend
patches = [mpatches.Patch(color=GRADE_COLORS[i], label=f'G{i+1}')
           for i in range(5)]
axes[1].legend(handles=patches, fontsize=5.5, loc='upper right', ncol=3)

# (c) Images per patient histogram
imgs_per_pat = split_df['Num_Images']
axes[2].hist(imgs_per_pat, bins=range(1, imgs_per_pat.max()+2),
             color=PAL['teal'], edgecolor='k', linewidth=0.3, alpha=0.8,
             align='left')
axes[2].set_xlabel('Images per Patient')
axes[2].set_ylabel('Count')
axes[2].set_title(f'(c) Images/Patient (μ={imgs_per_pat.mean():.1f})')

plt.tight_layout()
save_fig(fig, 'fig_bench_dataset_split')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 21 (Fig. D7): Comprehensive Ablation Heatmap - All Metrics
# ══════════════════════════════════════════════════════════════════════
# Single figure showing all key metrics for all ablations as a heatmap.

metric_keys = [
    ('prompted_dice_mean', 'Dice'),
    ('prompted_iou_mean', 'IoU'),
    ('mean_fg_dice', 'FG Dice'),
    ('grade_accuracy', 'Grade Acc'),
    ('grade_within1', 'Within ±1'),
    ('grade_mae', 'Grade MAE'),
    ('height_mae', 'Height MAE'),
    ('height_pearson_r', 'Height r'),
]

data_heat = np.zeros((4, len(metric_keys)))
for i, m in enumerate(abl_metrics):
    for j, (k, _) in enumerate(metric_keys):
        data_heat[i, j] = m[k]

# Normalise columns for color mapping (min-max per metric)
data_norm = data_heat.copy()
for j in range(data_heat.shape[1]):
    col = data_heat[:, j]
    mn, mx = col.min(), col.max()
    if mx > mn:
        data_norm[:, j] = (col - mn) / (mx - mn)
    else:
        data_norm[:, j] = 0.5
# Invert MAE columns (lower is better)
for j_inv in [5, 6]:  # grade_mae, height_mae
    data_norm[:, j_inv] = 1 - data_norm[:, j_inv]

fig, ax = plt.subplots(figsize=(6.5, 2.2))
im = ax.imshow(data_norm, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

for i in range(4):
    for j in range(len(metric_keys)):
        val = data_heat[i, j]
        # Format based on scale
        fmt = f'{val:.3f}' if val < 10 else f'{val:.2f}'
        ax.text(j, i, fmt, ha='center', va='center', fontsize=7,
                fontweight='bold' if data_norm[i, j] > 0.8 else 'normal')

ax.set_xticks(range(len(metric_keys)))
ax.set_xticklabels([l for _, l in metric_keys], fontsize=7.5, rotation=30, ha='right')
ax.set_yticks(range(4))
ax.set_yticklabels(variants)
ax.set_title('Ablation Study — All Metrics (Test Set)', pad=10)
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04, label='Normalised Score\n(↑ better)')

plt.tight_layout()
save_fig(fig, 'fig_bench_ablation_heatmap')
plt.show()
display(fig)

In [ ]:
# ══════════════════════════════════════════════════════════════════════

# Cell 24 (Fig. D8): Statistical Significance + Effect Size (Wilcoxon)

# ══════════════════════════════════════════════════════════════════════

df_stat = stat_df.copy()

df_stat = df_stat[df_stat['metric'] == 'prompted_dice'].copy()

df_stat['pair'] = df_stat['reference'] + ' vs ' + df_stat['compared']

df_stat['neglog10_p'] = -np.log10(df_stat['p_value'].clip(lower=1e-300))



fig, axes = plt.subplots(1, 2, figsize=(7, 2.8), gridspec_kw={'width_ratios': [1.1, 1]})



# (a) Mean Dice difference with CI-like visual emphasis via lollipop bars.

pairs = df_stat['pair'].tolist()

y = np.arange(len(df_stat))

axes[0].hlines(y, 0, df_stat['mean_diff'], color=PAL['grey'], lw=1.0, alpha=0.8)

axes[0].scatter(df_stat['mean_diff'], y, s=40, c=PAL['blue'], edgecolors='k', linewidth=0.4, zorder=3)

for i, row in df_stat.reset_index(drop=True).iterrows():

    axes[0].text(row['mean_diff'] + 0.005, i, f"{row['mean_diff']:.3f}", va='center', fontsize=7)

axes[0].axvline(0, color='k', lw=0.7, ls='--', alpha=0.6)

axes[0].set_yticks(y)

axes[0].set_yticklabels(pairs, fontsize=7)

axes[0].set_xlabel('Mean Dice Difference')

axes[0].set_title('(a) Effect Size (A3 reference)')



# (b) Significance as -log10(p), with alpha=0.05 threshold line.

bars = axes[1].bar(y, df_stat['neglog10_p'], color=PAL['green'], edgecolor='k', linewidth=0.4, alpha=0.8)

alpha_thr = -np.log10(0.05)

axes[1].axhline(alpha_thr, color=PAL['red'], ls='--', lw=1.0, label='p=0.05')

for i, row in df_stat.reset_index(drop=True).iterrows():

    axes[1].text(i, row['neglog10_p'] + 0.2, f"p={row['p_value']:.2e}", ha='center', va='bottom', fontsize=6, rotation=20)

axes[1].set_xticks(y)

axes[1].set_xticklabels(df_stat['compared'].tolist(), fontsize=7)

axes[1].set_ylabel(r'$-\log_{10}(p)$')

axes[1].set_title('(b) Wilcoxon Significance')

axes[1].legend(fontsize=6.5, loc='upper right')



plt.tight_layout()

save_fig(fig, 'fig_bench_statistical_significance')

plt.show()

display(fig)


## Section E - Summary Statistics & Verification

Final cell printing all key numbers for manuscript cross-checking.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 22: Summary Statistics for Manuscript Verification
# ══════════════════════════════════════════════════════════════════════
print('=' * 65)
print('  PROMPTRADSEG — MANUSCRIPT SUMMARY STATISTICS')
print('=' * 65)

a2m = [m for m in abl_metrics if m['ablation'] == 'A2'][0]

# Dataset
print(f'\n▸ Dataset')
print(f'  Total patients: {split_df.shape[0]}')
print(f'  Train/Val/Test patients: '
      f'{(split_df.Split=="train").sum()} / '
      f'{(split_df.Split=="val").sum()} / '
      f'{(split_df.Split=="test").sum()}')
print(f'  Test images: {len(df_a2)} (axial={n_ax}, sagittal={n_sag})')
print(f'  Graded samples (test): {a2m["n_graded"]}')

# Segmentation
print(f'\n▸ Segmentation (A2, Test)')
print(f'  Prompted Dice: {a2m["prompted_dice_mean"]:.3f} ± {a2m["prompted_dice_std"]:.3f}')
print(f'  Prompted IoU:  {a2m["prompted_iou_mean"]:.3f} ± {a2m["prompted_iou_std"]:.3f}')
print(f'  Mean FG Dice:  {a2m["mean_fg_dice"]:.3f}')
print(f'  Axial Dice:    {a2m["axial_prompted_dice"]:.3f}')
print(f'  Sagittal Dice: {a2m["sagittal_prompted_dice"]:.3f}')

# Per class
classes_ax = ['dice_IVD', 'dice_PE', 'dice_SC', 'dice_VB']
classes_sg = ['dice_IVD_sag', 'dice_Sacrum', 'dice_SC_sag']
print(f'  Per-class (axial): ' +
      ', '.join([f'{c.replace("dice_","")}={a2m[c]:.3f}' for c in classes_ax]))
print(f'  Per-class (sag):   ' +
      ', '.join([f'{c.replace("dice_","")}={a2m[c]:.3f}' for c in classes_sg]))

# Grading
print(f'\n▸ Grading (A2, Test)')
print(f'  Accuracy:  {a2m["grade_accuracy"]:.3f}')
print(f'  MAE:       {a2m["grade_mae"]:.3f}')
print(f'  Within ±1: {a2m["grade_within1"]:.3f}')
print(f'  Macro F1:  {cls_report["macro avg"]["f1-score"]:.3f}')
print(f'  Weighted F1: {cls_report["weighted avg"]["f1-score"]:.3f}')

# Disc height
print(f'\n▸ Disc Height (A2, Test)')
print(f'  MAE:      {a2m["height_mae"]:.3f} px')
print(f'  Pearson r: {a2m["height_pearson_r"]:.3f}')

# Efficiency
print(f'\n▸ Computational')
print(f'  Total params:     {phase2_info["total_params"]:,}')
print(f'  Trainable params: {a2m["trainable_params"]:,} '
      f'({a2m["trainable_params"]/phase2_info["total_params"]*100:.1f}%)')
print(f'  Peak VRAM: {a2m["vram_gb"]:.3f} GB')

# Statistical tests
print(f'\n▸ Statistical Tests (Wilcoxon)')
for _, row in stat_df.iterrows():
    print(f'  {row["reference"]} vs {row["compared"]}: '
          f'W={row["wilcoxon_stat"]:.0f}, p={row["p_value"]:.2e}')

# Naive baseline
graded = df_a2[df_a2['gt_grade'].isin([1,2,3,4,5])]
maj = graded['gt_grade'].mode()[0]
naive_acc = (graded['gt_grade'] == maj).mean()
naive_w1  = (abs(graded['gt_grade'] - maj) <= 1).mean()
print(f'\n▸ Naive Baseline (always predict Grade {maj})')
print(f'  Accuracy: {naive_acc:.3f}')
print(f'  Within ±1: {naive_w1:.3f}')

# List all saved figures
print(f'\n▸ Figures saved to: {FIGURES}')
figs_saved = sorted(glob.glob(os.path.join(FIGURES, '*.png')))
for fp in figs_saved:
    print(f'  {os.path.basename(fp)}')
print(f'\n  Total: {len(figs_saved)} PNG files')
print('=' * 65)

In [ ]:
# =============================================================================
# SECTION F - Main Manuscript Figures (Fig 6-11)
# Figs 1-5 are produced by visualize.py (or require TikZ/model inference);
# display cells show the pre-rendered output from figures_display/.
# =============================================================================


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Fig 1-5: Display pre-rendered figures (generated by visualize.py / TikZ)
# ──────────────────────────────────────────────────────────────────────
for stem in ['fig1_architecture', 'fig2_training_curves', 'fig3_qualitative',
             'fig4_ablation_bars', 'fig5_confusion_matrix']:
    print(f'\n── {stem} ──')
    show(stem)


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# Fig 6: Dataset Distribution  (fig6_dataset_distribution)
# 3-panel: (A) Pfirrmann grade distribution (test, grades 1-5 only),
#          (B) Imaging plane split (test),
#          (C) Patient split (train/val/test)
# ══════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(7.16, 2.6))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.42)
ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2])

# ── Panel A: Grade distribution from test CSV (valid grades 1-5 only) ──
grade_col = next((c for c in ['grade_gt', 'pfirrmann_grade', 'gt_grade', 'grade']
                  if c in df_a2.columns), None)
if grade_col:
    grade_counts = df_a2[df_a2[grade_col].isin([1, 2, 3, 4, 5])][grade_col].value_counts().sort_index()
    x_g = np.arange(len(grade_counts))
    grade_labels_a = [f'G{int(g)}' for g in grade_counts.index]
    bars_a = ax0.bar(x_g, grade_counts.values,
                     color=[GRADE_COLORS[i % len(GRADE_COLORS)] for i in range(len(grade_counts))],
                     edgecolor='white', linewidth=0.5, width=0.65)
    for bar, v in zip(bars_a, grade_counts.values):
        ax0.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + max(grade_counts.values) * 0.02,
                 str(v), ha='center', va='bottom', fontsize=7, fontweight='bold')
    ax0.set_xticks(x_g)
    ax0.set_xticklabels(grade_labels_a, fontsize=8)
    ax0.set_xlabel('Pfirrmann Grade')
    ax0.set_ylabel('Image Count')
    ax0.set_title('(A) Grade Distribution', fontsize=9, pad=5)
    ax0.set_ylim(0, grade_counts.max() * 1.18)
    ax0.spines['left'].set_linewidth(0.6)
    ax0.spines['bottom'].set_linewidth(0.6)
else:
    ax0.text(0.5, 0.5, 'grade column not found', transform=ax0.transAxes,
             ha='center', va='center', fontsize=8)
    ax0.set_title('(A) Grade Distribution', fontsize=9)

# ── Panel B: Plane split as annotated donut chart ────────────────────
plane_col = 'plane' if 'plane' in df_a2.columns else None
if plane_col:
    plane_counts = df_a2[plane_col].value_counts()
    plane_labels = [p.capitalize() for p in plane_counts.index]
    wedge_colors = [PAL['blue'], PAL['orange']]
    wedges, texts, autotexts = ax1.pie(
        plane_counts.values,
        labels=plane_labels,
        autopct='%1.0f%%',
        colors=wedge_colors,
        startangle=90,
        wedgeprops={'linewidth': 1.2, 'edgecolor': 'white', 'width': 0.6},
        textprops={'fontsize': 8},
        pctdistance=0.75,
        labeldistance=1.15,
    )
    for atext in autotexts:
        atext.set_fontsize(8)
        atext.set_fontweight('bold')
        atext.set_color('white')
    ax1.text(0, 0, f'n={n_test}', ha='center', va='center',
             fontsize=8, fontweight='bold', color='#333333')
    ax1.set_title('(B) Imaging Plane', fontsize=9, pad=10)
else:
    ax1.text(0.5, 0.5, 'plane column not found',
             transform=ax1.transAxes, ha='center', va='center', fontsize=8)
    ax1.set_title('(B) Imaging Plane', fontsize=9)

# ── Panel C: Patient split ────────────────────────────────────────────
split_cnts = {'Train': 0, 'Val': 0, 'Test': 0}
for _, row in split_df.iterrows():
    s = str(row.get('Split', '')).strip().title()
    if s in split_cnts:
        split_cnts[s] += 1
n_train, n_val, n_test_p = split_cnts['Train'], split_cnts['Val'], split_cnts['Test']
total_p = n_train + n_val + n_test_p or 1
split_x    = ['Train', 'Val', 'Test']
split_v    = [n_train, n_val, n_test_p]
split_cols = [PAL['blue'], PAL['orange'], PAL['green']]

bars_c = ax2.bar(split_x, split_v, color=split_cols,
                 edgecolor='white', linewidth=0.5, width=0.55)
for bar, v in zip(bars_c, split_v):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + total_p * 0.01,
             f'{v}\n({100*v/total_p:.0f}%)',
             ha='center', va='bottom', fontsize=7, fontweight='bold')
ax2.set_ylabel('Patients')
ax2.set_title('(C) Patient Split', fontsize=9, pad=5)
ax2.set_ylim(0, max(split_v) * 1.22)
ax2.spines['left'].set_linewidth(0.6)
ax2.spines['bottom'].set_linewidth(0.6)

plt.suptitle('LSMA-PQR Dataset — Distribution Summary', fontsize=10, y=1.03, fontweight='bold')
plt.tight_layout(pad=0.6)
save_fig(fig, 'fig6_dataset_distribution')
show('fig6_dataset_distribution')


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# Fig 7: Bland-Altman - IVD Height Agreement (fig7_bland_altman)
# Two panels: (A) Bland–Altman plot with confidence bands,
#             (B) GT vs Predicted scatter + regression line
# A6: pixel heights converted to mm using MEAN_SPACING_MM = 0.49
# ══════════════════════════════════════════════════════════════════════
MEAN_SPACING_MM = 0.49

gt_col   = next((c for c in ['gt_height', 'height_gt', 'height_true']
                 if c in df_a2.columns), None)
pred_col = next((c for c in ['pred_height', 'height_pred', 'height_predicted']
                 if c in df_a2.columns), None)

if gt_col and pred_col:
    ht      = df_a2[[gt_col, pred_col]].dropna()
    gt_v    = ht[gt_col].values.astype(float) * MEAN_SPACING_MM
    pred_v  = ht[pred_col].values.astype(float) * MEAN_SPACING_MM
    n_ht    = len(gt_v)
    mean_v  = (gt_v + pred_v) / 2.0
    diff_v  = pred_v - gt_v
    bias    = diff_v.mean()
    sd_d    = diff_v.std(ddof=1)
    se_bias = sd_d / np.sqrt(n_ht)          # SE of mean difference
    t_crit  = stats.t.ppf(0.975, n_ht - 1) # t for 95% CI
    bias_lo = bias - t_crit * se_bias
    bias_hi = bias + t_crit * se_bias
    loa_lo  = bias - 1.96 * sd_d
    loa_hi  = bias + 1.96 * sd_d
    # SE of LoA = sqrt(3/n) * sd
    se_loa  = np.sqrt(3 / n_ht) * sd_d
    mae_val = np.abs(diff_v).mean()
    r_val   = np.corrcoef(gt_v, pred_v)[0, 1]

    fig, axes = plt.subplots(1, 2, figsize=(7.16, 3.2))

    # ── (A) Bland–Altman ──────────────────────────────────────────────
    # Scatter with transparency
    axes[0].scatter(mean_v, diff_v, s=9, alpha=0.30, color=PAL['blue'],
                    linewidths=0, rasterized=True, zorder=2)
    # Reference zero
    axes[0].axhline(0, color='#aaaaaa', lw=0.5, ls=':', zorder=1)
    # Bias line + 95% CI band
    x_range = [mean_v.min() - 0.5, mean_v.max() + 0.5]
    axes[0].axhline(bias, color='k', lw=1.2, ls='-', zorder=3,
                    label=f'Bias = {bias:+.2f} mm')
    axes[0].fill_between(x_range, bias_lo, bias_hi,
                         color='k', alpha=0.12, zorder=2, label=f'95% CI bias')
    # LoA lines + CI bands
    for loa, lab in [(loa_hi, f'+1.96 SD = {loa_hi:+.2f}'),
                     (loa_lo, f'−1.96 SD = {loa_lo:+.2f}')]:
        axes[0].axhline(loa, color=PAL['red'], lw=0.9, ls='--', zorder=3, label=lab)
        axes[0].fill_between(x_range,
                             loa - t_crit * se_loa, loa + t_crit * se_loa,
                             color=PAL['red'], alpha=0.07, zorder=2)

    axes[0].set_xlim(x_range)
    axes[0].set_xlabel('Mean of GT and Predicted (mm)', fontsize=9)
    axes[0].set_ylabel('Predicted − GT (mm)', fontsize=9)
    axes[0].set_title('(A) Bland–Altman Agreement', fontsize=9, pad=5)
    axes[0].legend(fontsize=6.5, loc='upper right', ncol=2)
    axes[0].text(0.02, 0.04,
                 f'n = {n_ht}\nMAE = {mae_val:.2f} mm\nSD = {sd_d:.2f} mm',
                 transform=axes[0].transAxes, fontsize=7, va='bottom',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85,
                           edgecolor='#cccccc'))

    # ── (B) GT vs Predicted scatter + OLS line ────────────────────────
    axes[1].scatter(gt_v, pred_v, s=9, alpha=0.30, color=PAL['green'],
                    linewidths=0, rasterized=True, zorder=2)
    lims = [min(gt_v.min(), pred_v.min()) * 0.95,
            max(gt_v.max(), pred_v.max()) * 1.05]
    axes[1].plot(lims, lims, 'k--', lw=0.8, alpha=0.5, label='Identity', zorder=3)
    # OLS regression line
    slope, intercept, r_lr, p_lr, _ = stats.linregress(gt_v, pred_v)
    x_fit = np.array(lims)
    axes[1].plot(x_fit, slope * x_fit + intercept, color=PAL['orange'],
                 lw=1.2, ls='-', zorder=3,
                 label=f'OLS: y={slope:.2f}x+{intercept:.1f}')
    axes[1].set_xlim(lims)
    axes[1].set_ylim(lims)
    axes[1].set_aspect('equal')
    axes[1].set_xlabel('Ground Truth Height (mm)', fontsize=9)
    axes[1].set_ylabel('Predicted Height (mm)', fontsize=9)
    axes[1].set_title(f'(B) GT vs Predicted  (r = {r_val:.3f})', fontsize=9, pad=5)
    axes[1].legend(fontsize=6.5, loc='upper left')
    axes[1].text(0.98, 0.04,
                 f'r = {r_val:.3f}\np < 0.001',
                 transform=axes[1].transAxes, fontsize=7, va='bottom', ha='right',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85,
                           edgecolor='#cccccc'))

    plt.tight_layout(pad=0.5)
    save_fig(fig, 'fig7_bland_altman')
    show('fig7_bland_altman')
    print(f'Disc height — MAE: {mae_val:.3f} mm, Bias: {bias:.3f} mm, '
          f'LoA: [{loa_lo:.2f}, {loa_hi:.2f}], r: {r_val:.3f}')
else:
    print(f'Height columns not found. Available: {list(df_a2.columns[:25])}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Fig 8: Per-Disc-Level Segmentation Performance (fig8_per_disc_level)
# Bar chart (mean ± SD) with individual data points as jitter overlay
# ══════════════════════════════════════════════════════════════════════
level_col = next((c for c in ['disc_level', 'level', 'disc', 'spinal_level']
                  if c in df_a2.columns), None)
dice_col  = next((c for c in ['prompted_dice', 'dice', 'mean_dice']
                  if c in df_a2.columns), None)

rng = np.random.default_rng(42)

if level_col and dice_col:
    lvl_data = df_a2.groupby(level_col)[dice_col].agg(['mean','std','count']).sort_index()
    levels   = lvl_data.index.tolist()
    n_lvl    = len(levels)

    # Level-based color assignment (blues gradient) – dynamic palette
    cmap = plt.get_cmap('Blues')
    lvl_colors = [cmap((i + 1) / (n_lvl + 1)) for i in range(n_lvl)]

    fig, ax = plt.subplots(figsize=(max(3.8, n_lvl * 0.9 + 0.8), 3.2))
    x = np.arange(n_lvl)
    w = 0.55

    # Bars
    bars = ax.bar(x, lvl_data['mean'], w,
                  yerr=lvl_data['std'], capsize=4,
                  color=lvl_colors, edgecolor='white', linewidth=0.5,
                  error_kw={'linewidth': 0.9, 'ecolor': '#555555', 'capthick': 0.9},
                  alpha=0.80, zorder=2)

    # Individual data points (jitter strip)
    for i, lvl in enumerate(levels):
        vals = df_a2[df_a2[level_col] == lvl][dice_col].dropna().values
        if len(vals) > 0:
            jitter = rng.uniform(-w * 0.3, w * 0.3, len(vals))
            ax.scatter(x[i] + jitter, vals, s=5, alpha=0.35,
                       color=lvl_colors[i], edgecolors='none',
                       linewidths=0, zorder=3, rasterized=True)

    # Annotations
    for i, (_, row) in enumerate(lvl_data.iterrows()):
        y_top = row['mean'] + row['std'] + 0.045
        ax.text(i, min(y_top, 1.03),
                f'{row["mean"]:.3f}\n(n={int(row["count"])})',
                ha='center', va='bottom', fontsize=6.5, color='#333333')

    ax.set_xticks(x)
    ax.set_xticklabels([str(l) for l in levels], fontsize=8)
    ax.set_xlabel('Disc Level', fontsize=9)
    ax.set_ylabel('Prompted Dice  (mean ± SD)', fontsize=9)
    ax.set_title('Per-Disc-Level Segmentation Performance  (A2, Test)', fontsize=9, pad=6)
    ax.set_ylim(0, 1.18)
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.05))
    ax.tick_params(axis='y', which='minor', length=2, width=0.4)
    ax.spines['left'].set_linewidth(0.6)
    ax.spines['bottom'].set_linewidth(0.6)

    # Horizontal reference grid
    for y_ref in [0.5, 0.7, 0.9]:
        ax.axhline(y_ref, color='#dddddd', lw=0.5, ls='--', zorder=1)

    plt.tight_layout(pad=0.5)
    save_fig(fig, 'fig8_per_disc_level')
    show('fig8_per_disc_level')

else:
    # Fallback: plane-based grouping
    plane_col = 'plane' if 'plane' in df_a2.columns else None
    if dice_col and plane_col:
        groups = df_a2.groupby(plane_col)[dice_col].agg(['mean','std','count'])
        fig, ax = plt.subplots(figsize=(3.5, 3.0))
        x = np.arange(len(groups))
        bars = ax.bar(x, groups['mean'], yerr=groups['std'],
                      color=[PAL['blue'], PAL['orange']], edgecolor='white',
                      capsize=4, alpha=0.85)
        for i, (_, row) in enumerate(groups.iterrows()):
            vals = df_a2[df_a2[plane_col] == row.name][dice_col].dropna().values
            jitter = rng.uniform(-0.15, 0.15, len(vals))
            ax.scatter(x[i] + jitter, vals, s=5, alpha=0.3,
                       color=[PAL['blue'], PAL['orange']][i], rasterized=True)
        ax.set_xticks(x)
        ax.set_xticklabels(groups.index, fontsize=8)
        ax.set_ylabel('Mean Dice  (± SD)')
        ax.set_title('Dice by Plane (disc_level col not found)')
        ax.set_ylim(0, 1.12)
        plt.tight_layout(pad=0.5)
        save_fig(fig, 'fig8_per_disc_level')
        show('fig8_per_disc_level')
    else:
        print(f'Required columns not found. Available: {list(df_a2.columns)}')


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# Fig 9: ROC Curves - Grading (OvR) & Disorder Detection
# (fig9_roc_curves)
# ══════════════════════════════════════════════════════════════════════
from sklearn.preprocessing import label_binarize

fig, axes = plt.subplots(1, 2, figsize=(7.16, 3.2))

# ── Panel A: One-vs-Rest ROC for Pfirrmann grading ───────────────────
grade_col     = next((c for c in ['grade_gt', 'pfirrmann_grade', 'gt_grade', 'grade']
                      if c in df_a2.columns), None)
prob_cols     = sorted([c for c in df_a2.columns
                        if c.startswith('grade_prob_') or
                           c.startswith('prob_grade_')])

auc_vals_g = {}
if grade_col and prob_cols:
    grades_uniq = sorted([g for g in df_a2[grade_col].dropna().unique()
                          if not np.isnan(g)])
    y_true_bin  = label_binarize(df_a2[grade_col].fillna(-1), classes=grades_uniq)
    grade_probs = []
    for g in grades_uniq:
        matching = [c for c in prob_cols if str(int(g)) in c.split('_')]
        if matching:
            grade_probs.append(df_a2[matching[0]].fillna(0).values)
    if len(grade_probs) == len(grades_uniq):
        for i, g in enumerate(grades_uniq):
            fpr, tpr, _ = roc_curve(y_true_bin[:, i], grade_probs[i])
            auc_v = auc(fpr, tpr)
            auc_vals_g[g] = auc_v
            axes[0].plot(fpr, tpr, lw=1.1,
                         color=GRADE_COLORS[i % len(GRADE_COLORS)],
                         label=f'G{int(g)}  AUC={auc_v:.2f}')
            axes[0].fill_between(fpr, 0, tpr, alpha=0.04,
                                 color=GRADE_COLORS[i % len(GRADE_COLORS)])
        # Micro-average
        y_flat = y_true_bin.ravel()
        p_flat = np.concatenate(grade_probs)
        fpr_m, tpr_m, _ = roc_curve(y_flat, p_flat)
        auc_m = auc(fpr_m, tpr_m)
        axes[0].plot(fpr_m, tpr_m, 'k--', lw=1.2,
                     label=f'Micro avg  {auc_m:.2f}')
    else:
        axes[0].text(0.5, 0.5,
                     f'Grade probability columns not found\n({len(prob_cols)} prob cols detected)',
                     transform=axes[0].transAxes, ha='center', va='center', fontsize=8)
else:
    axes[0].text(0.5, 0.5,
                 'Grade probability columns not found\n(hard predictions only available)',
                 transform=axes[0].transAxes, ha='center', va='center', fontsize=8)

axes[0].plot([0, 1], [0, 1], ':', color='#aaaaaa', lw=0.8)
axes[0].set_xlim([-0.01, 1.01])
axes[0].set_ylim([-0.01, 1.06])
axes[0].set_xlabel('False Positive Rate', fontsize=9)
axes[0].set_ylabel('True Positive Rate', fontsize=9)
axes[0].set_title('(A) Grading ROC  (One-vs-Rest)', fontsize=9, pad=5)
axes[0].legend(fontsize=6.5, loc='lower right', title='AUC', title_fontsize=6.5)
axes[0].set_aspect('equal', adjustable='box')

# ── Panel B: Disorder detection ROC ──────────────────────────────────
disorder_specs = [
    ('Bulge',         'disorder_Bulge_prob',       'disorder_Bulge_gt'),
    ('Thecal Comp.',  'disorder_ThecalComp_prob',  'disorder_ThecalComp_gt'),
    ('Nerve Comp.',   'disorder_NerveComp_prob',   'disorder_NerveComp_gt'),
    ('Protrusion',    'disorder_Protrusion_prob',  'disorder_Protrusion_gt'),
    ('Herniation',    'disorder_Herniation_prob',  'disorder_Herniation_gt'),
    ('Stenosis',      'disorder_Stenosis_prob',    'disorder_Stenosis_gt'),
]
d_colors = [PAL['blue'], PAL['green'], PAL['orange'],
            PAL['red'],  PAL['purple'], PAL['teal']]

plotted_d = 0
for (name, pcol, gcol), c in zip(disorder_specs, d_colors):
    if pcol not in df_a2.columns or gcol not in df_a2.columns:
        continue
    mask = df_a2[[gcol, pcol]].dropna().index
    y_t  = df_a2.loc[mask, gcol].values
    y_p  = df_a2.loc[mask, pcol].values
    if len(np.unique(y_t)) < 2:
        continue
    fpr_d, tpr_d, _ = roc_curve(y_t, y_p)
    auc_d = auc(fpr_d, tpr_d)
    axes[1].plot(fpr_d, tpr_d, lw=1.1, color=c,
                 label=f'{name}  {auc_d:.2f}')
    axes[1].fill_between(fpr_d, 0, tpr_d, alpha=0.04, color=c)
    plotted_d += 1

if plotted_d == 0:
    axes[1].text(0.5, 0.5, 'Disorder probability\ncolumns not found',
                 transform=axes[1].transAxes, ha='center', va='center', fontsize=8)

axes[1].plot([0, 1], [0, 1], ':', color='#aaaaaa', lw=0.8)
axes[1].set_xlim([-0.01, 1.01])
axes[1].set_ylim([-0.01, 1.06])
axes[1].set_xlabel('False Positive Rate', fontsize=9)
axes[1].set_ylabel('True Positive Rate', fontsize=9)
axes[1].set_title('(B) Disorder Detection ROC', fontsize=9, pad=5)
axes[1].legend(fontsize=6.5, loc='lower right', title='AUC', title_fontsize=6.5)
axes[1].set_aspect('equal', adjustable='box')

plt.tight_layout(pad=0.8)
save_fig(fig, 'fig9_roc_curves')
show('fig9_roc_curves')


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# Fig 10: Per-Grade Classification Performance (fig10_grade_performance)
# Grouped bars (F1/Prec/Rec) + secondary axis for sample count
# ══════════════════════════════════════════════════════════════════════
grade_labels_10 = ['1', '2', '3', '4', '5']
f1_vals, prec_vals, rec_vals, support_vals = [], [], [], []

for g in grade_labels_10:
    d = cls_report.get(g, cls_report.get(f'grade_{g}',
                       cls_report.get(f'Grade {g}',
                       cls_report.get(f'Grade{g}', {}))))
    f1_vals.append(d.get('f1-score',  0.0))
    prec_vals.append(d.get('precision', 0.0))
    rec_vals.append(d.get('recall',    0.0))
    support_vals.append(int(d.get('support', 0)))

wa = cls_report.get('weighted avg', {})
ma = cls_report.get('macro avg', {})
acc_overall = cls_report.get('accuracy', 0.0)

fig, ax1 = plt.subplots(figsize=(5.0, 3.2))
x     = np.arange(len(grade_labels_10))
width = 0.23

bars_f1   = ax1.bar(x - width, f1_vals,   width, label='F1',
                    color=PAL['blue'],   edgecolor='white', lw=0.4, alpha=0.90)
bars_prec = ax1.bar(x,         prec_vals, width, label='Precision',
                    color=PAL['green'],  edgecolor='white', lw=0.4, alpha=0.90)
bars_rec  = ax1.bar(x + width, rec_vals,  width, label='Recall',
                    color=PAL['orange'], edgecolor='white', lw=0.4, alpha=0.90)

ax1.set_xlabel('Pfirrmann Grade', fontsize=9)
ax1.set_ylabel('Score', fontsize=9)
ax1.set_title('Per-Grade Classification Metrics  (A2, Test)', fontsize=9, pad=5)
ax1.set_xticks(x)
ax1.set_xticklabels([f'G{g}' for g in grade_labels_10], fontsize=8)
ax1.set_ylim(0, 1.22)
ax1.legend(fontsize=7, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.01))
ax1.spines['left'].set_linewidth(0.6)
ax1.spines['bottom'].set_linewidth(0.6)

# F1 annotations on bars
for bar, v in zip(bars_f1, f1_vals):
    if v > 0.03:
        ax1.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.013,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=6.5,
                 color=PAL['blue'], fontweight='bold')

# Recall annotations
for bar, v in zip(bars_rec, rec_vals):
    if v > 0.03:
        ax1.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.013,
                 f'{v:.2f}', ha='center', va='bottom', fontsize=5.5,
                 color='#555555')

# Secondary y-axis: sample counts as diamonds
ax2 = ax1.twinx()
ax2.spines['right'].set_visible(True)
ax2.spines['right'].set_linewidth(0.6)
ax2.plot(x, support_vals, 'D-', color=PAL['red'],
         ms=4.5, lw=0.9, label='Support (n)', zorder=4)
for xi, v in zip(x, support_vals):
    ax2.text(xi + 0.14, v + max(support_vals) * 0.03,
             str(v), fontsize=6, color=PAL['red'], ha='left', va='bottom')
ax2.set_ylabel('Support (n)', color=PAL['red'], fontsize=8)
ax2.tick_params(axis='y', labelcolor=PAL['red'], labelsize=7)
ax2.set_ylim(0, max(support_vals) * 1.6 + 5)
ax2.legend(loc='upper right', fontsize=7, bbox_to_anchor=(1.0, 0.78))

# Summary stats box
ax1.text(0.02, 0.97,
         f'Weighted F1 = {wa.get("f1-score", 0):.3f}\n'
         f'Macro F1    = {ma.get("f1-score", 0):.3f}\n'
         f'Accuracy    = {acc_overall:.3f}',
         transform=ax1.transAxes, fontsize=7, va='top',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.88,
                   edgecolor='#cccccc'))

plt.tight_layout(pad=0.5)
save_fig(fig, 'fig10_grade_performance')
show('fig10_grade_performance')


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# Fig 11: Failure Case Analysis (fig11_failure_analysis)
# 3 panels: (A) Dice histogram - all vs failures; (B) failure rate by
# imaging plane; (C) failure rate by Pfirrmann grade (grades 1-5 only)
# ══════════════════════════════════════════════════════════════════════
dice_col   = next((c for c in ['prompted_dice', 'dice', 'mean_dice']
                   if c in df_a2.columns), None)
grade_col  = next((c for c in ['grade_gt', 'pfirrmann_grade', 'gt_grade', 'grade']
                   if c in df_a2.columns), None)
plane_col  = 'plane' if 'plane' in df_a2.columns else None

FAIL_THR = 0.5   # Dice < 0.5 → failure

fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.9))

# ── Panel A: Dice density - all vs failures ──────────────────────────
if dice_col:
    all_dice  = df_a2[dice_col].dropna()
    fail_dice = all_dice[all_dice < FAIL_THR]
    fail_rate = len(fail_dice) / len(all_dice) * 100 if len(all_dice) > 0 else 0
    bins = np.linspace(0, 1, 35)
    axes[0].hist(all_dice, bins=bins, color=PAL['blue'],  alpha=0.55,
                 density=True, label=f'All  (n={len(all_dice)})')
    axes[0].hist(fail_dice, bins=bins, color=PAL['red'],  alpha=0.80,
                 density=True, label=f'Fail Dice<{FAIL_THR}  (n={len(fail_dice)})')
    axes[0].axvline(FAIL_THR, color=PAL['red'],  lw=1.0, ls='--')
    axes[0].axvline(all_dice.median(), color='k', lw=0.8, ls=':',
                    label=f'Median={all_dice.median():.3f}')
    axes[0].set_xlabel('Prompted Dice', fontsize=9)
    axes[0].set_ylabel('Density', fontsize=9)
    axes[0].set_title('(A) Dice Distribution', fontsize=9, pad=5)
    axes[0].legend(fontsize=6.5)
    axes[0].text(0.02, 0.97, f'Failure rate: {fail_rate:.1f}%',
                 transform=axes[0].transAxes, fontsize=7.5, va='top',
                 color=PAL['red'], fontweight='bold')
else:
    axes[0].text(0.5, 0.5, 'dice col not found',
                 transform=axes[0].transAxes, ha='center', va='center')

# ── Panel B: Failure rate by imaging plane ────────────────────────────
if dice_col and plane_col:
    plane_groups = df_a2.groupby(plane_col)[dice_col]
    p_names, p_rates, p_ns = [], [], []
    for name, grp in plane_groups:
        g = grp.dropna()
        rate = (g < FAIL_THR).sum() / len(g) * 100 if len(g) > 0 else 0
        p_names.append(name.capitalize())
        p_rates.append(rate)
        p_ns.append(len(g))
    bar_cols = [PAL['blue'], PAL['orange']][:len(p_names)]
    b2 = axes[1].bar(p_names, p_rates, color=bar_cols, edgecolor='white',
                     lw=0.5, width=0.55, alpha=0.88)
    for xi, (v, n) in enumerate(zip(p_rates, p_ns)):
        axes[1].text(xi, v + max(p_rates) * 0.03 + 0.2,
                     f'{v:.1f}%\n(n={n})', ha='center', fontsize=7.5,
                     fontweight='bold')
    axes[1].set_ylabel('Failure Rate (%)', fontsize=9)
    axes[1].set_title('(B) Failure by Plane', fontsize=9, pad=5)
    axes[1].set_ylim(0, max(p_rates) * 1.55 + 1)
    axes[1].axhline(fail_rate if dice_col else 0, color='k',
                    lw=0.7, ls=':', alpha=0.5, label='Overall')
    axes[1].legend(fontsize=6.5)
else:
    axes[1].text(0.5, 0.5, 'plane/dice not found',
                 transform=axes[1].transAxes, ha='center', va='center')

# ── Panel C: Failure rate by Pfirrmann grade (grades 1-5 only) ───────
if dice_col and grade_col:
    # Filter to valid grades only (exclude -1 from sagittal unlabelled images)
    df_graded = df_a2[df_a2[grade_col].isin([1, 2, 3, 4, 5])]
    grade_fail_r, grade_fail_n = {}, {}
    for g, grp in df_graded.groupby(grade_col)[dice_col]:
        g_clean = grp.dropna()
        grade_fail_r[int(g)] = (g_clean < FAIL_THR).sum() / len(g_clean) * 100
        grade_fail_n[int(g)] = len(g_clean)
    g_keys  = sorted(grade_fail_r.keys())
    g_rates = [grade_fail_r[g] for g in g_keys]
    g_ns    = [grade_fail_n[g] for g in g_keys]
    bars_c  = axes[2].bar([f'G{g}' for g in g_keys], g_rates,
                           color=[GRADE_COLORS[i % len(GRADE_COLORS)]
                                  for i in range(len(g_keys))],
                           edgecolor='white', lw=0.5, width=0.55, alpha=0.88)
    for bar, v, n in zip(bars_c, g_rates, g_ns):
        axes[2].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + max(g_rates) * 0.03 + 0.2,
                     f'{v:.0f}%\nn={n}', ha='center', fontsize=6.5,
                     fontweight='bold')
    axes[2].set_ylabel('Failure Rate (%)', fontsize=9)
    axes[2].set_title('(C) Failure by Grade', fontsize=9, pad=5)
    axes[2].set_ylim(0, (max(g_rates) if g_rates else 50) * 1.55 + 1)
    axes[2].axhline(fail_rate if dice_col else 0, color='k',
                    lw=0.7, ls=':', alpha=0.5, label='Overall')
    axes[2].legend(fontsize=6.5)
else:
    axes[2].text(0.5, 0.5, 'grade/dice not found',
                 transform=axes[2].transAxes, ha='center', va='center')

plt.tight_layout(pad=0.8)
save_fig(fig, 'fig11_failure_analysis')
show('fig11_failure_analysis')
